# Historical experiment notebook
This notebook preserves its original protocol and embedded source. For offline result reproduction use `scripts/reproduce.py` in the repository. For a new measurement keep the original experiment identity and do not overwrite old results. Archive checks are intentionally strict.


# CPU–GPU Heterogeneous Inference Lab

Build and measure your own static, layer-wise GPT-2 CPU/CUDA runtime.

**What runs here:** explicit transformer blocks, manual activation transfers, device-local KV caches, correctness checks, prefill and cached-decode benchmarks, copy-path measurements, and plots. Everything is FP32 for the baseline. There is no automatic device dispatch or hidden offloading library.

**Before running:** select a GPU runtime and inspect its actual device name. Report T4 results only when the allocated GPU is a T4. The initial sweep is intentionally small; optional larger experiments are disabled below.

**Build-time status:** 39 CPU software tests passed and 9 CUDA tests were skipped in the authoring environment. No pretrained/T4 numbers have been prefilled. This notebook runs the missing integration checks on your actual runtime.

The notebook is self-contained: the first code cell writes the project source to the runtime. Inspect `hetero/engine.py` and `hetero/model.py` in the file browser, or use the accompanying ZIP to study/edit the code.

## 1. Write the project files

This cell unpacks the embedded **source code only**. It does not download or bundle model weights. Existing files are preserved so rerunning this cell does not overwrite your edits. For a clean reset, use a new runtime or a different `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import base64, io, os, sys, zipfile

PROJECT_ROOT = Path("/content/heterogeneous_inference") if Path("/content").exists() else Path.cwd() / "heterogeneous_inference"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE_B64))) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        if not target.is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("Unsafe archive member")
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        elif not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
os.chdir(PROJECT_ROOT)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Project:", PROJECT_ROOT)
print("Read README.md for architecture, measurement definitions, and limitations.")


## 2. Install dependencies

Keep Colab's CUDA-enabled PyTorch. The pinned Transformers dependency is used only as an independent correctness reference, not by the custom inference engine. Run from a fresh runtime before importing these libraries.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt", "-r", "requirements-reference.txt"], check=True)


## 3. Check hardware and choose run settings

Each command below executes in a separate Python process so the reference model and demonstration do not leave live model tensors in the benchmark process. The CPU thread count and exact model snapshot are recorded.

In [ ]:
import json
from datetime import datetime, timezone
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Select a GPU runtime, then rerun the notebook.")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("GPU capacity GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
if "T4" not in GPU_NAME:
    print("This is not a T4. The experiment can run, but label results with this actual GPU.")

MODEL_ID = "openai-community/gpt2"
MODEL_REVISION = "main"  # Replaced by an exact snapshot SHA after validation.
CPU_THREADS = min(2, os.cpu_count() or 1)
RUN_ROOT = Path("results") / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print("Run output:", RUN_ROOT.resolve())


## 4. Software correctness tests

On a CUDA runtime, this includes the nine device-placement/cache-locality tests that were skipped in the CPU-only authoring environment. Do not continue after a failure.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## 5. Independent pretrained GPT-2 validation

This is the first checkpoint download. Compare all-position prompt logits and a teacher-forced cached chunk against Hugging Face. It checks CPU, GPU, prefix, suffix and interleaved placements. Maximum errors and top-1 agreement are saved; floating-point equality is not assumed.

If this gate fails, inspect the error before collecting performance numbers. Do not merely loosen tolerances to make a failure disappear.

In [ ]:
validation_path = RUN_ROOT / "validation.json"
subprocess.run([sys.executable, "-m", "hetero.validate",
                "--model", MODEL_ID, "--revision", MODEL_REVISION,
                "--threads", str(CPU_THREADS), "--out", str(validation_path)], check=True)
validation = json.loads(validation_path.read_text())
assert validation["checks"] and all(row["passed"] for row in validation["checks"])
MODEL_REVISION = validation["model"]["revision"]
print("Pinned checkpoint revision:", MODEL_REVISION)
import pandas as pd
from IPython.display import display, Image

display(pd.DataFrame(validation["checks"]))


## 6. Initial placement sweep

Six cases: 0/6/12 GPU blocks, batch 1, prompt lengths 32/128, and 8 generated tokens. One warmup and three repeats are a functional experiment, **not** a strong tail-latency estimate.

`cpu_offload_ratio = (12 - gpu_layers) / 12`. Mixed plans retain the tied embedding/output weight on GPU, so the block fraction is not the VRAM fraction. Input/output token copies and hidden-state copies are distinguished.

In [ ]:
quick_dir = RUN_ROOT / "quick"
subprocess.run([sys.executable, "-m", "hetero.benchmark",
                "--model", MODEL_ID, "--revision", MODEL_REVISION,
                "--gpu-layers", "0", "6", "12", "--layouts", "prefix",
                "--batches", "1", "--seq-lens", "32", "128",
                "--new-tokens", "8", "--warmup", "1", "--repeats", "3",
                "--threads", str(CPU_THREADS), "--out", str(quick_dir)], check=True)
summary = pd.read_csv(quick_dir / "summary.csv")
columns = ["gpu_layers", "cpu_offload_ratio", "batch_size", "sequence_length", "status",
           "prefill_ms_median", "ttft_ms_median", "decode_tpot_ms_median",
           "decode_generated_tokens_per_s_median", "generation_generated_tokens_per_s_median",
           "gpu_peak_allocated_bytes", "activation_boundaries"]
display(summary[[c for c in columns if c in summary.columns]])


## 7. Plot actual measurements

The plotting script makes separate figures for prefill, TTFT, cached decode, end-to-end generated throughput, GPU memory, and a **separate serialized-profile** transfer fraction. Input-token throughput is never labeled generation speed.

In [ ]:
subprocess.run([sys.executable, "-m", "hetero.plot", str(quick_dir / "summary.csv")], check=True)
for name in ["B1_S128_decode_generated_tokens_per_s_median.png",
             "B1_S128_gpu_peak_allocated_MiB.png"]:
    image = quick_dir / "figures" / name
    if image.exists():
        display(Image(filename=str(image)))


## 8. Preallocated copy-path microbenchmark

Measure H2D and D2H separately with pinned and pageable host memory. Allocation and pinning are outside the timed interval. CUDA event spans and synchronized host wall times are both saved.

These are effective copy-path measurements, not direct physical PCIe line-rate measurements. Tiny transfers can be dominated by fixed overhead. Nothing here claims copy/compute overlap, and the baseline model engine still uses blocking `.to()` transfers.

In [ ]:
copy_dir = RUN_ROOT / "copies"
subprocess.run([sys.executable, "-m", "hetero.copy_bench", "--out", str(copy_dir),
                "--repeats", "50", "--warmup", "5"], check=True)
display(pd.read_csv(copy_dir / "copy_summary.csv"))


## 9. Larger offload/batch/prompt sweep — optional

Enable after the first gates and small run succeed. This is 45 configurations, with fixed precision and endpoint placement. Increase repetitions for selected final comparisons and retain raw trial data. The script checkpoints CSVs after each case and records CUDA OOM cases.

Existing output directories are not overwritten: reruns should use a new directory or a new `RUN_ROOT`.

In [ ]:
RUN_FULL_SWEEP = False
if RUN_FULL_SWEEP:
    main_dir = RUN_ROOT / "main"
    subprocess.run([sys.executable, "-m", "hetero.benchmark",
                    "--model", MODEL_ID, "--revision", MODEL_REVISION,
                    "--gpu-layers", "0", "3", "6", "9", "12",
                    "--layouts", "prefix", "--batches", "1", "2", "4",
                    "--seq-lens", "32", "128", "512", "--new-tokens", "32",
                    "--warmup", "3", "--repeats", "10", "--threads", str(CPU_THREADS),
                    "--out", str(main_dir)], check=True)
    subprocess.run([sys.executable, "-m", "hetero.plot", str(main_dir / "summary.csv")], check=True)
else:
    print("Larger sweep disabled. Set RUN_FULL_SWEEP = True to execute it.")


## 10. Same block count, different boundaries — optional

At six GPU blocks, prefix and suffix have two hidden-state device crossings. The interleaved placement has twelve. Compare measured bytes, profile copy time, prefill, and cached decode while holding the block count fixed.

In [ ]:
RUN_LAYOUT_COMPARISON = False
if RUN_LAYOUT_COMPARISON:
    layout_dir = RUN_ROOT / "layouts"
    subprocess.run([sys.executable, "-m", "hetero.benchmark",
                    "--model", MODEL_ID, "--revision", MODEL_REVISION,
                    "--gpu-layers", "6", "--layouts", "prefix", "suffix", "interleaved",
                    "--batches", "1", "4", "--seq-lens", "32", "512",
                    "--new-tokens", "32", "--warmup", "3", "--repeats", "10",
                    "--threads", str(CPU_THREADS), "--out", str(layout_dir)], check=True)
    subprocess.run([sys.executable, "-m", "hetero.plot", str(layout_dir / "summary.csv")], check=True)
else:
    print("Layout comparison disabled. Set RUN_LAYOUT_COMPARISON = True to execute it.")


## 11. Choose among tested placements under a memory budget

This selects from measured rows for **one fixed workload**, not across incompatible batch/sequence lengths. The budget applies to peak PyTorch allocation, not all physical VRAM used by every process. The example budget is an experimental constraint, not a measured performance claim.

In [ ]:
BATCH = 1
PROMPT_LENGTH = 128
GPU_BUDGET_MIB = 512
candidates = summary[
    (summary["status"] == "ok") &
    (summary["batch_size"] == BATCH) &
    (summary["sequence_length"] == PROMPT_LENGTH) &
    (summary["gpu_peak_allocated_bytes"] <= GPU_BUDGET_MIB * 2**20)
].sort_values("generation_generated_tokens_per_s_median", ascending=False)
if candidates.empty:
    print("No successful measured placement meets that budget/workload.")
else:
    display(candidates[["gpu_layers", "cpu_offload_ratio", "gpu_peak_allocated_bytes",
                        "generation_generated_tokens_per_s_median", "ttft_ms_median"]])
    print("Top row is the best measured end-to-end generated throughput within this budget.")


## 12. Generate text with the custom engine — optional

This demonstration is outside all benchmarks. GPT-2 text quality is not the point of the hardware experiment. Its output is produced by the custom model and offloading loop, not by Hugging Face `generate()`.

The local function releases the demo model afterward; do not retain a second live GPU model while benchmarking.

In [ ]:
RUN_TEXT_DEMO = False
if RUN_TEXT_DEMO:
    def text_demo():
        from huggingface_hub import hf_hub_download
        from tokenizers import Tokenizer
        from hetero import GPT2, OffloadEngine
        model = GPT2.from_pretrained(MODEL_ID, MODEL_REVISION)
        engine = OffloadEngine(model, gpu_layers=6, layout="prefix")
        tokenizer = Tokenizer.from_file(hf_hub_download(
            MODEL_ID, "tokenizer.json", revision=MODEL_REVISION))
        prompt = "The future of artificial intelligence is"
        ids = torch.tensor([tokenizer.encode(prompt).ids], dtype=torch.long)
        output = engine.generate_fixed(ids, new_tokens=24)
        return tokenizer.decode(ids[0].tolist() + output[0].tolist())
    print(text_demo())
    import gc
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Text demo disabled. Set RUN_TEXT_DEMO = True to run it.")


## 13. Save results and write the report

Keep `validation.json`, metadata, raw trials, summaries, profiles, copy measurements and package freeze. Use `REPORT_TEMPLATE.md`. The generated ZIP contains actual results from **this runtime**, not prefilled example numbers.

In [ ]:
import shutil
(RUN_ROOT / "environment.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
shutil.copyfile("REPORT_TEMPLATE.md", RUN_ROOT / "REPORT_TEMPLATE.md")
archive_path = shutil.make_archive(str(RUN_ROOT) + "_export", "zip", root_dir=RUN_ROOT)
print("Saved:", Path(archive_path).resolve())
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Open the saved ZIP from your notebook's file browser.")


## Interpretation checklist

The interesting finding is a measured memory/latency tradeoff, not a guaranteed speedup over GPU-only execution. Keep these distinctions in your write-up:

- Static CPU layer execution is not weight streaming or a full FlexGen reproduction.
- Offloaded block fraction is not total parameter or peak-memory fraction; the tied embedding/output weight has an explicit placement policy.
- Prefill input-token throughput, cached generated-token throughput, and end-to-end generation throughput are different metrics.
- Serialized stage profiles are diagnostic; they are not uninstrumented end-to-end timings or pure PCIe DMA measurements.
- One dependent batch does not overlap CPU and GPU layer work merely because it uses both devices.

Read `README.md` for the cost model, sources, scope, and suggested follow-up experiments.